# Rarity Review
Runs `Rarity_Scorer` to rank tagged pages by how far their composition deviates from their structural category, then displays the flagged pages' `class_map` for direct visual inspection — the human-review step in the workflow. This is a triage heuristic, not a statistical test (see `Rarity_Scorer` docstrings).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from xrf.config import Xrf_Comparison_Config
from xrf.signatures.leaf_signature import Leaf_Signature_Extractor
from xrf.comparison.category_registry import Category_Registry
from xrf.comparison.rarity_scoring import Rarity_Scorer
from xrf.visualization.xrf_plots import Build_Category_Montage

comparison_config = Xrf_Comparison_Config()

In [ ]:
# Project directory and data paths
PROJECT_DIR = Path.cwd()

# Find project root directory
while not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "xrf"
OUTPUT_DATA_DIR = DATA_DIR / "output"
PROCESSED_DATA_DIR = OUTPUT_DATA_DIR / "processed"
FIGURES_DIR = OUTPUT_DATA_DIR / "figures"
COMPARISON_DIR = OUTPUT_DATA_DIR / "comparison"
MONTAGES_DIR = COMPARISON_DIR / "montages"

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
MONTAGES_DIR.mkdir(parents=True, exist_ok=True)

## Load per-page signatures and categories
Only tagged pages are included — untagged pages are skipped.

In [ ]:
Page_Signatures = {}
Page_Categories = {}

for Meta_Path in sorted(PROCESSED_DATA_DIR.glob("page_*_meta.json")):
    Page_Id = Meta_Path.stem.replace("_meta", "")
    Category = Category_Registry.Load_Page_Category(Meta_Path)

    if Category is None:
        print(f"Skipping {Page_Id}: not yet tagged (run 05_page_categorization.ipynb).")
        continue

    with open(Meta_Path, "r") as Meta_File:
        Meta = json.load(Meta_File)
    Optimal_K = Meta["optimal_k"]

    Labels = np.load(PROCESSED_DATA_DIR / f"{Page_Id}_labels.npy")
    Page_Signatures[Page_Id] = Leaf_Signature_Extractor.Compute_Abundances(
        Labels, Num_Classes=Optimal_K
    )
    Page_Categories[Page_Id] = Category

print(f"Loaded {len(Page_Signatures)} tagged page(s).")

## Rank pages by rarity
Categories need at least two pages with non-zero MAD in every class to be scored — `Rarity_Scorer.Compute_Robust_Deviation` raises `RuntimeError` rather than silently producing inf/nan when a category's MAD is exactly zero for a class.

In [ ]:
Rankings = Rarity_Scorer.Rank_Pages_By_Rarity(Page_Signatures, Page_Categories, comparison_config)

Rarity_Scores = [
    {"page_id": Page_Id, "max_abs_deviation": Deviation, "is_flagged": Is_Flagged}
    for Page_Id, Deviation, Is_Flagged in Rankings
]

with open(COMPARISON_DIR / "rarity_scores.json", "w") as Scores_File:
    json.dump(Rarity_Scores, Scores_File, indent=2)

Rarity_Scores

## Review flagged pages
Display the `class_map` for every page flagged as rare, for direct human review.

In [ ]:
Flagged_Page_Ids = [Page_Id for Page_Id, _, Is_Flagged in Rankings if Is_Flagged]

if not Flagged_Page_Ids:
    print("No pages flagged at the current Rarity_Mad_Threshold "
          f"({comparison_config.Rarity_Mad_Threshold}).")

for Page_Id in Flagged_Page_Ids:
    Class_Map_Path = PROCESSED_DATA_DIR / f"{Page_Id}_class_map.npy"
    if not Class_Map_Path.exists():
        print(f"No class_map found for {Page_Id}.")
        continue

    Class_Map = np.load(Class_Map_Path)
    plt.figure(figsize=(6, 6))
    plt.imshow(Class_Map, cmap="tab10")
    plt.title(f"{Page_Id} — flagged rare ({Page_Categories[Page_Id]})")
    plt.axis("off")
    plt.show()

## Montage of flagged pages (illustrative)
Assembles the shared book-level `Cluster_k_Visual.png` files for a quick side-by-side review of flagged pages. Revisit once per-page cluster exports exist.

In [ ]:
Cluster_Visual_Paths = sorted(FIGURES_DIR.glob("Cluster_*_Visual.png"))

if Flagged_Page_Ids and Cluster_Visual_Paths:
    Build_Category_Montage(
        Cluster_Visual_Paths,
        MONTAGES_DIR / "flagged_pages_review_montage.png",
        Grid_Cols=4,
    )
else:
    print("Skipping montage: no flagged pages and/or no cluster visuals available.")